# sciences po

## récupération des chercheurs

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {"User-Agent": "Mozilla/5.0"}

urls = [
    "https://www.sciencespo.fr/department-economics/people/permanent-faculty/",
    "https://www.sciencespo.fr/department-economics/people/permanent-faculty/1/"
]

data = []

for url in urls:

    response = requests.get(url, headers=headers)
    response.encoding = "utf-8"
    html = response.text

    soup = BeautifulSoup(html, "html.parser")

    cards = soup.find_all("div", class_="research-item-module--researcherCardList--a9250")

    for card in cards:

        name_tag = card.find("h3")
        title_tag = card.find("p", class_="research-item-module--title--094b0")

        name = name_tag.get_text(" ", strip=True) if name_tag else None
        title = title_tag.get_text(" ", strip=True) if title_tag else None

        # récupérer les research interests
        interests = None
        interest_block = card.find("p", class_="research-item-module--containerAll--8bb35")

        if interest_block:
            spans = interest_block.find_all("span")
            if len(spans) > 1:
                interests = spans[1].get_text(" ", strip=True)

        data.append({
            "name": name,
            "title_raw": title,
            "research_interests": interests
        })

df = pd.DataFrame(data)

print(df)

                        name  \
0                 Yann Algan   
1          Facundo Argañaraz   
2            Ghazala Azmat †   
3             Johannes Boehm   
4             Paul Bouscasse   
5                 Julia Cagé   
6               Pierre Cahuc   
7             Giulia Caprini   
8        Nicolas Coeurdacier   
9     Pierre-Philippe Combes   
10           Jeanne Commault   
11   Clément De Chaisemartin   
12             Antoine Ferey   
13           Axelle Ferriere   
14          Roberto Galbiati   
15          Stéphane Guibaud   
16             Sergei Guriev   
17          Jeanne Hagenbach   
18                 Junnan He   
19              Emeric Henry   
20           Kerstin Holzheu   
21            Clément Imbert   
22           Suzanna Khalifa   
23        Justine Knebelmann   
24             Marleen Marra   
25             Thierry Mayer   
26           Isabelle Mejean   
27         Claire Montialoux   
28  Kevin Hjortshøj O'Rourke   
29            Franz Ostrizek   
30      

In [2]:
df

,name,title_raw,research_interests
0,Yann Algan,Professor,Public Economics; Cultural Economics; Social I...
1,Facundo Argañaraz,Assistant Professor,Econometrics; Causal Inference; Semiparametric...
2,Ghazala Azmat †,Professor,Labour Economics; Public Economics; Gender Eco...
3,Johannes Boehm,Associate Professor (on secondment),Macroeconomics; International Trade; Industria...
4,Paul Bouscasse,Assistant Professor,Macroeconomics; Monetary Economics; Internatio...
5,Julia Cagé,Professor,Democracy; Information Economics; Political Ec...
6,Pierre Cahuc,Professor,"Unemployment, Macroeconomics, Labour Markets"
7,Giulia Caprini,Assistant Professor,Political Economy; Health Economics; Behaviour...
8,Nicolas Coeurdacier,Professor,"International Macroeconomics and Finance, Macr..."
9,Pierre-Philippe Combes,"CNRS Research Professor , Head of doctoral stu...",International TradeEconomic GeographyCities an...


In [3]:
df["title_raw"].value_counts()

title_raw
Professor                                                          15
Assistant Professor                                                12
CNRS Research Professor                                             3
Assistant Professor (on secondment)                                 3
CNRS Research Professor , Head of doctoral studies in Economics     1
Associate Professor (on secondment)                                 1
CNRS Professor                                                      1
Professor (on secondment)                                           1
Professor , Head of the Department of Economics                     1
CNRS Research Professor , OFCE Chairman                             1
Name: count, dtype: int64

# création de la classifcation et du rang

In [4]:
def classify_position(title):

    t = title.lower()

    # Assistant Professor
    if "assistant professor" in t:
        return "Assistant Professor", "B"

    # Associate Professor
    if "associate professor" in t:
        return "Associate Professor", "A"

    # CNRS (à traiter plus tard)
    if "cnrs" in t:
        return "CNRS", None

    # Professor (mais pas assistant/associate)
    if "professor" in t:
        return "Full Professor", "A"

    return "Other", None

In [5]:
df[["profession", "rang"]] = df["title_raw"].apply(
    lambda x: pd.Series(classify_position(x))
)

In [6]:
df

,name,title_raw,research_interests,profession,rang
0,Yann Algan,Professor,Public Economics; Cultural Economics; Social I...,Full Professor,A
1,Facundo Argañaraz,Assistant Professor,Econometrics; Causal Inference; Semiparametric...,Assistant Professor,B
2,Ghazala Azmat †,Professor,Labour Economics; Public Economics; Gender Eco...,Full Professor,A
3,Johannes Boehm,Associate Professor (on secondment),Macroeconomics; International Trade; Industria...,Associate Professor,A
4,Paul Bouscasse,Assistant Professor,Macroeconomics; Monetary Economics; Internatio...,Assistant Professor,B
5,Julia Cagé,Professor,Democracy; Information Economics; Political Ec...,Full Professor,A
6,Pierre Cahuc,Professor,"Unemployment, Macroeconomics, Labour Markets",Full Professor,A
7,Giulia Caprini,Assistant Professor,Political Economy; Health Economics; Behaviour...,Assistant Professor,B
8,Nicolas Coeurdacier,Professor,"International Macroeconomics and Finance, Macr...",Full Professor,A
9,Pierre-Philippe Combes,"CNRS Research Professor , Head of doctoral stu...",International TradeEconomic GeographyCities an...,CNRS,None


## traitement des chercheurs cnrs dénommés 'CNRS Research Professor': 

In [7]:
pip install unidecode

Note: you may need to restart the kernel to use updated packages.


In [8]:
import unidecode
import re

def build_profile_url(name):

    parts = name.lower().split()

    first = parts[0]
    last = parts[-1]

    # enlever accents
    first = unidecode.unidecode(first)
    last = unidecode.unidecode(last)

    # remplacer apostrophes par tiret
    last = last.replace("'", "-")

    # enlever autres caractères spéciaux
    first = re.sub(r"[^a-z\-]", "", first)
    last = re.sub(r"[^a-z\-]", "", last)

    slug = f"{last}-{first}"

    url = f"https://www.sciencespo.fr/department-economics/directory/{slug}/"

    return url

In [9]:
def classify_cnrs_page(url):

    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        response = requests.get(url, headers=headers)
        response.encoding = "utf-8"

        soup = BeautifulSoup(response.text, "html.parser")

        bio = soup.find("div", class_="ckContent")

        text = bio.get_text(" ", strip=True).lower()

        if "director" in text or "directeur de recherche" in text or "(dr)" in text:
            return "DR CNRS", "A"

        elif "cnrs" in text:
            return "CR CNRS", "B"

        else:
            return "CNRS", None

    except:
        return "CNRS", None
    

In [10]:
for i, row in df.iterrows():

    if "cnrs" in row["title_raw"].lower():

        url = build_profile_url(row["name"])

        profession, rang = classify_cnrs_page(url)

        df.loc[i, "profession"] = profession
        df.loc[i, "rang"] = rang

In [11]:
df

,name,title_raw,research_interests,profession,rang
0,Yann Algan,Professor,Public Economics; Cultural Economics; Social I...,Full Professor,A
1,Facundo Argañaraz,Assistant Professor,Econometrics; Causal Inference; Semiparametric...,Assistant Professor,B
2,Ghazala Azmat †,Professor,Labour Economics; Public Economics; Gender Eco...,Full Professor,A
3,Johannes Boehm,Associate Professor (on secondment),Macroeconomics; International Trade; Industria...,Associate Professor,A
4,Paul Bouscasse,Assistant Professor,Macroeconomics; Monetary Economics; Internatio...,Assistant Professor,B
5,Julia Cagé,Professor,Democracy; Information Economics; Political Ec...,Full Professor,A
6,Pierre Cahuc,Professor,"Unemployment, Macroeconomics, Labour Markets",Full Professor,A
7,Giulia Caprini,Assistant Professor,Political Economy; Health Economics; Behaviour...,Assistant Professor,B
8,Nicolas Coeurdacier,Professor,"International Macroeconomics and Finance, Macr...",Full Professor,A
9,Pierre-Philippe Combes,"CNRS Research Professor , Head of doctoral stu...",International TradeEconomic GeographyCities an...,DR CNRS,A


In [12]:
df_sciencespo = df

In [13]:
df_sciencespo = df_sciencespo.rename(columns={"title_raw": "appelation_sur_site"})

In [14]:
df_sciencespo 

,name,appelation_sur_site,research_interests,profession,rang
0,Yann Algan,Professor,Public Economics; Cultural Economics; Social I...,Full Professor,A
1,Facundo Argañaraz,Assistant Professor,Econometrics; Causal Inference; Semiparametric...,Assistant Professor,B
2,Ghazala Azmat †,Professor,Labour Economics; Public Economics; Gender Eco...,Full Professor,A
3,Johannes Boehm,Associate Professor (on secondment),Macroeconomics; International Trade; Industria...,Associate Professor,A
4,Paul Bouscasse,Assistant Professor,Macroeconomics; Monetary Economics; Internatio...,Assistant Professor,B
5,Julia Cagé,Professor,Democracy; Information Economics; Political Ec...,Full Professor,A
6,Pierre Cahuc,Professor,"Unemployment, Macroeconomics, Labour Markets",Full Professor,A
7,Giulia Caprini,Assistant Professor,Political Economy; Health Economics; Behaviour...,Assistant Professor,B
8,Nicolas Coeurdacier,Professor,"International Macroeconomics and Finance, Macr...",Full Professor,A
9,Pierre-Philippe Combes,"CNRS Research Professor , Head of doctoral stu...",International TradeEconomic GeographyCities an...,DR CNRS,A


# AMSE

## extraction du nom des chercheurs

In [15]:
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

base_url = "https://www.amse-aixmarseille.fr/fr/membres?field_pr_member_type_target_id=29&field_pr_department_target_id=All&field_pr_full_name_value=&page="

names = []

for page in range(0,8):

    url = base_url + str(page)

    response = requests.get(url, headers=headers)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    blocks = soup.find_all("div", class_="views-field-field-pr-last-name")

    for block in blocks:

        name = block.get_text(" ", strip=True)
        names.append(name)

print(names)

['Mohammad Abu-Zaineh', 'Marcel Aloy', 'Frédéric Aprahamian', 'Dominique Augey', 'Patricia Augier', 'Guillaume Bazot', 'Mohamed Belhaj', 'Nicolas Berman', 'Pierre Bertrand', 'Sebastian Bervoets', 'Vincent Bignon', 'Raouf Boucekkine', 'Renaud Bourlès', 'Yann Bramoullé', 'Olivier Chanel', 'Nicolas Clootens', 'Jernej Copic', 'Russell Davidson', 'Marion Davin', 'Bruno Decreuse', 'Timothée Demont', 'Frédéric Deroïan', 'Habiba Djebbari', 'Marion Dovis', 'Frédéric Dufourt', 'Gilles Dufrénot', 'Alice Fabre', 'Mathieu Faure', 'Lisandro Fermin', 'Romain Ferrali', 'Nathalie Ferrière', 'Charles Figuières', 'Emmanuel Flachaire', 'Marco Fongoni', 'Gaëtan Fournier', 'Ewen Gallic', 'Cecilia Garcia Peñalosa', 'Pierre Garello', 'Karine Gente', 'Antoine Gentier', 'Badih Ghattas', 'Céline Gimet', 'Eric Girardin', 'Charles Gottlieb', 'Yajna Govind', 'Pierre Granier', 'Nicolas Gravel', 'Arthur Guillouzouic', 'Houda Hafidi', 'Julien Hanoteau', 'Dominique Henriet', 'Fanny Henriet', 'Sullivan Hué', 'Hélène Lau

In [16]:
import pandas as pd

df_amse = pd.DataFrame(names, columns=["name"])

print(df_amse)

                   name
0   Mohammad Abu-Zaineh
1           Marcel Aloy
2   Frédéric Aprahamian
3       Dominique Augey
4       Patricia Augier
..                  ...
83  Tanguy van Ypersele
84       Alain Venditti
85       Bruno Ventelou
86         Jiakun Zheng
87       Roberta Ziparo

[88 rows x 1 columns]


In [17]:
import unidecode
import re

def build_amse_url(name):

    parts = name.split()

    first = parts[0]
    last_parts = parts[1:]   # tout le reste = nom de famille

    last = "-".join(last_parts)

    # enlever accents
    last = unidecode.unidecode(last.lower())

    # enlever caractères spéciaux
    last = re.sub(r"[^a-z\-]", "", last)

    url = f"https://www.amse-aixmarseille.fr/fr/membres/{last}"

    return url

## scrapping de la bio des chercheurs

In [18]:
def scrape_amse_profile(url):

    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    # name
    name_tag = soup.find("div", class_="field--name-field-pr-full-name")
    name = name_tag.get_text(strip=True) if name_tag else None

    # fonction locale (Chercheur / Professeur / etc.)
    role = None
    affiliation = None

    role_block = soup.find("div", class_="fs-5")

    if role_block:
        spans = role_block.find_all("span")
        if len(spans) >= 2:
            role = spans[0].get_text(strip=True)
            affiliation = spans[1].get_text(strip=True)

    # statut
    statut = None
    statut_block = soup.find("dt", string="Statut")
    if statut_block:
        statut = statut_block.find_next("dd").get_text(" ", strip=True)

    # research fields
    research = None
    research_block = soup.find("dt", string="Domaine(s) de recherche")
    if research_block:
        research = research_block.find_next("dd").get_text(" ", strip=True)

    # phd
    phd = None
    phd_block = soup.find("dt", string="Thèse")
    if phd_block:
        phd = phd_block.find_next("dd").get_text(" ", strip=True)

    # email
    email = None
    mail = soup.select_one("a[href^='mailto']")
    if mail:
        email = mail.get_text(strip=True)

    return {
        "name": name,
        "role": role,
        "affiliation": affiliation,
        "statut": statut,
        "research_fields": research,
        "phd": phd,
        "email": email
    }

In [19]:
amse_data = []

for name in df_amse["name"]:

    url = build_amse_url(name)

    profile = scrape_amse_profile(url)

    amse_data.append(profile)

df_amse_profiles = pd.DataFrame(amse_data)

In [20]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [21]:
df_amse_profiles

,name,role,affiliation,statut,research_fields,phd,email
0,Mohammad Abu-Zaineh,Chercheur,Aix-Marseille Université,Professeur des universités,"Économétrie, Économie de la santé, Économie publique","2008, Aix-Marseille Université",mohammad.abu-zaineh[at]univ-amu.fr
1,Marcel Aloy,Chercheur,Aix-Marseille Université,Maître de conférences,"Économétrie, Macroéconomie",None,marcel.aloy[at]univ-amu.fr
2,Frédéric Aprahamian,Chercheur,Université de Toulon,Maître de conférences,"Économie comportementale et expérimentale, Économétrie, Économie de l'environnement","1991, Aix-Marseille Université",frederic.aprahamian[at]univ-amu.fr
3,Dominique Augey,Chercheuse,Aix-Marseille Université,Professeur des universités,"Philosophie économique, Économie publique","1995, Aix-Marseille Université",dominique.augey[at]univ-amu.fr
4,Patricia Augier,Chercheuse,Aix-Marseille Université,Professeur des universités,"Économie du développement, Économie internationale et géographique","1995, Aix-Marseille Université",patricia.augier[at]univ-amu.fr
5,Guillaume Bazot,Chercheur,Aix-Marseille Université,Professeur des universités,"Finance, Macroéconomie","2011, École des hautes études en sciences sociales",guillaume.bazot[at]univ-amu.fr
6,Mohamed Belhaj,Chercheur,Centrale Méditerranée,Professeur,"Finance, Théorie des jeux et réseaux sociaux","2005, University of Toulouse 1",mohamed.belhaj[at]centrale-med.fr
7,Nicolas Berman,Chercheur,CNRS,Directeur de recherche,"Économie du développement, Économie internationale et géographique","2008, Université Paris 1 Panthéon-Sorbonne",nicolas.berman[at]univ-amu.fr
8,Philippe Bertrand,Membre affilié,Aix-Marseille Université,Professeur des universités,Finance,None,philippe.bertrand[at]univ-amu.fr
9,Sebastian Bervoets,Chercheur,CNRS,Chargé de recherche,"Théorie des jeux et réseaux sociaux, Choix social","2005, Aix-Marseille Université",sebastian.bervoets[at]univ-amu.fr


In [22]:
df_amse_profiles["statut"].value_counts()

statut
Professeur des universités                 38
Maître de conférences                      24
Directeur de recherche                      9
Chargé de recherche                         5
Professeur émérite                          4
Professeur                                  3
Professeur associé                          2
Chaire de professeur junior en économie     1
Directeur de recherche émérite              1
Directeur d’études émérite                  1
Name: count, dtype: int64

## classification chercheur statut amses

In [23]:
def classify_amse(statut, affiliation):

    s = statut.lower() if statut else ""
    a = affiliation.lower() if affiliation else ""

    # rang A
    if "professeur des universités" in s:
        return "Professeur des universités", "A"

    if s == "professeur":
        return "Full Professor", "A"

    if "professeur associé" in s:
        return "Associate Professor", "A"

    if "directeur de recherche" in s:
        if "cnrs" in a:
            return "DR CNRS", "A"
        if "ehess" in a:
            return "DR EHESS", "A"
        return "DR", "A"

    # rang B
    if "maître de conférences" in s:
        if "ehess" in a:
            return "Maître de conférences EHESS", "B"
        return "Maître de conférences", "B"

    if "chargé de recherche" in s:
        if "cnrs" in a:
            return "CR CNRS", "B"
        if "ehess" in a:
            return "CR EHESS", "B"
        return "CR", "B"

    if "chaire de professeur junior" in s:
        return "Assistant Professor", "B"

    return None, None

In [24]:
df_amse_profiles[["profession", "rang"]] = df_amse_profiles.apply(
    lambda row: pd.Series(classify_amse(row["statut"], row["affiliation"])),
    axis=1
)

In [25]:
df_amse_profiles

,name,role,affiliation,statut,research_fields,phd,email,profession,rang
0,Mohammad Abu-Zaineh,Chercheur,Aix-Marseille Université,Professeur des universités,"Économétrie, Économie de la santé, Économie publique","2008, Aix-Marseille Université",mohammad.abu-zaineh[at]univ-amu.fr,Professeur des universités,A
1,Marcel Aloy,Chercheur,Aix-Marseille Université,Maître de conférences,"Économétrie, Macroéconomie",None,marcel.aloy[at]univ-amu.fr,Maître de conférences,B
2,Frédéric Aprahamian,Chercheur,Université de Toulon,Maître de conférences,"Économie comportementale et expérimentale, Économétrie, Économie de l'environnement","1991, Aix-Marseille Université",frederic.aprahamian[at]univ-amu.fr,Maître de conférences,B
3,Dominique Augey,Chercheuse,Aix-Marseille Université,Professeur des universités,"Philosophie économique, Économie publique","1995, Aix-Marseille Université",dominique.augey[at]univ-amu.fr,Professeur des universités,A
4,Patricia Augier,Chercheuse,Aix-Marseille Université,Professeur des universités,"Économie du développement, Économie internationale et géographique","1995, Aix-Marseille Université",patricia.augier[at]univ-amu.fr,Professeur des universités,A
5,Guillaume Bazot,Chercheur,Aix-Marseille Université,Professeur des universités,"Finance, Macroéconomie","2011, École des hautes études en sciences sociales",guillaume.bazot[at]univ-amu.fr,Professeur des universités,A
6,Mohamed Belhaj,Chercheur,Centrale Méditerranée,Professeur,"Finance, Théorie des jeux et réseaux sociaux","2005, University of Toulouse 1",mohamed.belhaj[at]centrale-med.fr,Full Professor,A
7,Nicolas Berman,Chercheur,CNRS,Directeur de recherche,"Économie du développement, Économie internationale et géographique","2008, Université Paris 1 Panthéon-Sorbonne",nicolas.berman[at]univ-amu.fr,DR CNRS,A
8,Philippe Bertrand,Membre affilié,Aix-Marseille Université,Professeur des universités,Finance,None,philippe.bertrand[at]univ-amu.fr,Professeur des universités,A
9,Sebastian Bervoets,Chercheur,CNRS,Chargé de recherche,"Théorie des jeux et réseaux sociaux, Choix social","2005, Aix-Marseille Université",sebastian.bervoets[at]univ-amu.fr,CR CNRS,B


In [26]:
df_amse_profiles["affiliation"].value_counts()

affiliation
Aix-Marseille Université                  57
CNRS                                      15
Centrale Méditerranée                      6
Sciences Po Aix                            4
EHESS                                      2
Banque de France                           1
Université de Toulon                       1
Faculté d'économie et de gestion (FEG)     1
KEDGE Business School                      1
Name: count, dtype: int64

In [27]:
df_amse_profiles = df_amse_profiles[~df_amse_profiles["statut"].str.contains("émérite", case=False, na=False)]

In [28]:
df_amse_profiles = df_amse_profiles[~df_amse_profiles["affiliation"].str.contains("KEDGE Business School", case=False, na=False)]

In [29]:
df_amse_profiles = df_amse_profiles[~df_amse_profiles["affiliation"].str.contains("Banque de France", case=False, na=False)]

In [30]:
df_amse_profiles["profession"].value_counts()

profession
Professeur des universités     38
Maître de conférences          23
DR CNRS                         9
CR CNRS                         5
Full Professor                  3
Assistant Professor             1
Maître de conférences EHESS     1
Name: count, dtype: int64

In [31]:
df_amse_profiles["profession"].unique()

array(['Professeur des universités', 'Maître de conférences',
       'Full Professor', 'DR CNRS', 'CR CNRS', 'Assistant Professor',
       'Maître de conférences EHESS'], dtype=object)

# on crée le genre à partir de la variable rôle qui donne chercheuse ou chercheur

In [32]:
df_amse_profiles["gender"] = df_amse_profiles["role"].map({
    "Chercheuse": "Female",
    "Chercheur": "Male"
})

In [33]:
df_amse_profiles["gender"].value_counts()

gender
Male      62
Female    17
Name: count, dtype: int64

In [34]:
gender_by_prof = (
    df_amse_profiles
    .groupby(["profession", "gender"])
    .size()
    .unstack(fill_value=0)
)

In [35]:
1+1

2

In [36]:
gender_by_prof["share_female"] = (
    gender_by_prof["Female"] /
    (gender_by_prof["Female"] + gender_by_prof["Male"])
)

In [37]:
gender_by_prof

gender,Female,Male,share_female
profession,,,
Assistant Professor,1,0,1.000000
CR CNRS,0,5,0.000000
DR CNRS,0,9,0.000000
Full Professor,0,3,0.000000
Maître de conférences,5,18,0.217391
Maître de conférences EHESS,1,0,1.000000
Professeur des universités,10,27,0.270270


# TSE

## récupération du nom des chercheurs

In [38]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {"User-Agent": "Mozilla/5.0"}

data = []

for page in range(0,20):

    url = f"https://www.tse-fr.eu/fr/people/1777/-1/-1?page={page}"

    response = requests.get(url, headers=headers)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    rows = soup.select("div.views-row")

    if len(rows) == 0:
        break

    for row in rows:

        name = row.select_one(".cell.name")
        name = name.get_text(strip=True) if name else None

        cells = row.select(".cell")

        title = None
        role = None

        if len(cells) >= 3:
            title = cells[2].get_text(strip=True)

        if len(cells) >= 4:
            role = cells[3].get_text(strip=True)

        link = row.select_one("a.line")

        profile_url = None
        if link:
            profile_url = "https://www.tse-fr.eu" + link["href"]

        data.append({
            "name": name,
            "appelation_site": title,
            "role": role,
            "profile_url": profile_url
        })

df_tse = pd.DataFrame(data)

print(df_tse)
print("Nombre de chercheurs :", len(df_tse))

                          name  \
0         Angélique Acquatella   
1                  David Alary   
2                 Ingela Alger   
3    Bénédicte Alziary Chassat   
4                 Stefan Ambec   
5          Jean-Pierre Amigues   
6             Henrik Andersson   
7              Tiziana Assenza   
8                 Andrea Attar   
9            Emmanuelle Auriol   
10              Jean-Paul Azam   
11             Victoria Barham   
12               Pascal Bégout   
13                Milo Bianchi   
14          Christophe Bisière   
15             Adrien Blanchet   
16                Matteo Bobba   
17                Jérôme Bolte   
18      Jean-François Bonnefon   
19               Céline Bonnet   
20          Christian Bontemps   
21            Philippe Bontems   
22    Zohra Bouamra-Mechemache   
23      Michel-Benoit Bouissou   
24             Rachid Boumahdi   
25            Matthieu Bouvard   
26    Marie-Françoise Calmette   
27            Stéphane Caprice   
28         Cat

In [148]:
df_tse["appelation_site"].value_counts()

appelation_site
Maître de conférences, TSE                                                               14
Professeur, TSE                                                                          11
Professeur d'Economie, TSE                                                                9
Professeur assistant, TSE                                                                 9
Directeur de recherche, CNRS                                                              6
Professeure assistante, TSE                                                               4
Professeur de Mathématiques, TSE                                                          4
Chargé de recherche, CNRS                                                                 3
Directeur de recherche, INRAe                                                             3
Maître de conférences, Université Toulouse Capitole                                       2
Professeur d'économie                                           

# nettoyage des données en enlevant émerité, Finance, et sciences politiques

In [40]:
df_tse = df_tse[~df_tse["appelation_site"].str.contains("émérite", case=False, na=False)]

In [41]:
df_tse = df_tse[~df_tse["appelation_site"].str.contains("Emérite", case=False, na=False)]

In [42]:
df_tse = df_tse[~df_tse["appelation_site"].str.contains("Finance", case=False, na=False)]

In [43]:
df_tse = df_tse[~df_tse["appelation_site"].str.contains("science politiques", case=False, na=False)]

In [147]:
df_tse = df_tse[~df_tse["appelation_site"].str.contains("sciences politiques", case=False, na=False)]

In [139]:
def classify_tse(title):

    t = title.lower() if title else ""

    # DR
    if ("directeur de recherche" in t or 
        "directrice de recherche" in t or
        "directeur de recherches" in t or
        "directrice de recherches" in t) and "cnrs" in t:
        return "DR CNRS", "A"

    if ("directeur de recherche" in t or 
        "directrice de recherche" in t) and "inrae" in t:
        return "DR INRAE", "A"

    # CR
    if ("chargé de recherche" in t or 
        "chargée de recherche" in t or
        "chargé de recherches" in t or
        "chargée de recherches" in t) and "cnrs" in t:
        return "CR CNRS", "B"

    if ("chargé de recherche" in t or 
        "chargée de recherche" in t) and "inrae" in t:
        return "CR INRAE", "B"

    # Assistant Professor
    if ("assistant professeur" in t or
        "assistante professeur" in t or
        "professeur assistant" in t or
        "professeure assistante" in t):
        return "Assistant Professor", "B"

    # Maître de conférences
    if ("maître de conférences" in t or 
        "maîtresse de conférences" in t):
        return "Maître de conférences", "B"

    return None, None

In [140]:
df_tse[["profession","rang"]] = df_tse["appelation_site"].apply(
    lambda x: pd.Series(classify_tse(x))
)

In [141]:
len(df_tse[df_tse["profession"].notna()])

60

### on s'occupe de ceux qui n'ont pas encore été classifiés

In [47]:
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0"}

def get_cv_link(url):

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    cv_tag = soup.find("a", class_="lien pdf")

    if cv_tag:
        return cv_tag["href"]

    return None

In [48]:
def download_cv(cv_url, name):

    r = requests.get(cv_url, headers=headers)

    with open(f"cv/{name}.pdf", "wb") as f:
        f.write(r.content)

## récupération des cv en pdf

In [49]:
import os

os.makedirs("cv", exist_ok=True)

In [50]:
for i, row in df_tse[df_tse["profession"].isna()].iterrows():

    url = row["profile_url"]

    cv_url = get_cv_link(url)

    if cv_url:
        download_cv(cv_url, row["name"])

### analyse des cvs

In [51]:
pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [78]:
import pdfplumber

In [79]:
def extract_text_from_pdf(path):

    text = ""

    try:
        with pdfplumber.open(path) as pdf:

            for page in pdf.pages:

                page_text = page.extract_text()

                if page_text:
                    text += page_text + "\n"

    except:
        return None

    return text

In [80]:
def clean_text(text):

    text = text.lower()

    text = text.replace("assistantprofessor", "assistant professor")
    text = text.replace("associateprofessor", "associate professor")
    text = text.replace("fullprofessor", "full professor")

    text = text.replace("toulouseschoolofeconomics", "toulouse school of economics")

    return text

In [81]:
import re

def extract_current_position(text):

    lines = text.lower().split("\n")

    keywords = [
        "assistant professor",
        "directeur d'études", 
        "directeur d'etudes", 
        "associate professor",
        "chargée de recherche",
        "directrice de recherche",
        "professeure",
        "professor",
        "professeur",
        "research director",
        "directeur de recherche",
        "chargé de recherche",
        "maître de conférences",
        "researcher"
    ]

    # on regarde seulement le début du CV
    for line in lines[:20]:

        if "@" in line:
            continue

        if any(k in line for k in keywords):
            return line.strip()

    return None

In [82]:
def extract_year(line):

    match = re.search(r"(19|20)\d{2}", line)

    if match:
        return match.group()

    return None

In [83]:
def extract_past_positions(text):

    lines = text.lower().split("\n")

    positions = []

    keywords = [
        "assistant professor",
        "associate professor",
        "professor",
        "directeur d'études", 
        "directeur d'etudes",
        "maître de conférences",
        "chargé de recherche",
        "directeur de recherche",
        "director of research",
        "professeur",
        "research director",
        "researcher",
        "postdoc"
    ]

    for line in lines:

        if len(line) < 15:
            continue

        if not re.search(r"(19|20)\d{2}", line):
            continue

        if any(k in line for k in keywords):

            year_match = re.search(r"(19|20)\d{2}", line)
            year = int(year_match.group())

            positions.append({
                "year": year,
                "text": line.strip()
            })

    return positions

In [84]:
def infer_rank_from_past_positions(text):

    text = text.lower()

    if "associate professor" in text:
        return "A"

    if "full professor" in text:
        return "A"

    if "assistant professor" in text:
        return "B"

    return None

In [85]:
def detect_prof_qualification(text):

    text = text.lower()

    if "qualification aux fonctions de professeur" in text:
        return "A"
    return None

In [87]:
def extract_birth_year(text):

    match = re.search(r"(born|birth).*?(19|20)\d{2}", text.lower())

    if match:
        return match.group()

    return None

In [88]:
def extract_phd_year(text):

    text = text.lower()

    match = re.search(r'phd[^0-9]{0,20}(19|20)\d{2}', text)

    if match:
        year = re.search(r'(19|20)\d{2}', match.group())
        return int(year.group())

    return None

In [89]:
def classify_rank(position):

    if position is None:
        return None

    if position in [
        "Full Professor",
        "Professeur des universités",
        "Associate Professor",
        "Maître de conférences",
        "DR CNRS / INRAE"
    ]:
        return "A"

    if position in [
        "Assistant Professor",
        "CR CNRS / INRAE"
    ]:
        return "B"

    return None

In [90]:
def correct_rank(current_position, past_positions):

    text = ""

    if current_position:
        text += current_position.lower()

    if past_positions:
        text += " ".join([p["text"] for p in past_positions]).lower()

    # rang A
    if (
        "research director" in text
        or "directeur de recherche" in text
        or "professor" in text
        or "professeur" in text
        or "associate professor" in text
        or "maître de conférences" in text
    ):
        return "A"

    # rang B
    if (
        "assistant professor" in text
        or "chargé de recherche" in text
    ):
        return "B"

    return None

In [91]:
def extract_position(text):

    if text is None:
        return None

    text = text.lower()

    if "directeur de recherche" in text or "directrice de recherche" in text or "research director" in text or "director of research" in text:
        return "DR CNRS / INRAE"
    
    if "directeur d’études" in text or "directeur d'etudes" in text:
        return "Directeur d'études (EHESS)"

    if "chargé de recherche" in text:
        return "CR CNRS / INRAE"
    if "chargée de recherche" in text:
        return "CR CNRS / INRAE"

    if "senior researcher" in text:
         return "DR CNRS / INRAE"

    if "junior researcher" in text:
        return "CR CNRS / INRAE"

    if "assistant professor" in text:
        return "Assistant Professor"

    if "associate professor" in text:
        return "Associate Professor"

    if "maître de conférences" in text:
        return "Maître de conférences"

    if "professeur" in text or "professeure" in text:
        return "Professeur des universités"

    if "professor" in text:
        return "Full Professor"

    return None

In [92]:
def get_latest_position(past_positions):

    if not past_positions:
        return None

    valid = []

    for p in past_positions:

        if p["year"] is None:
            continue

        text = p["text"].lower()

        # ignorer postdocs / visiting
        if "post doc" in text or "postdoc" in text or "visiting" in text:
            continue

        valid.append(p)

    if not valid:
        return None

    latest = max(valid, key=lambda x: x["year"])

    return extract_position(latest["text"])

In [93]:
def analyse_cv(path):

    text = extract_text_from_pdf(path)
    if text is None:
        return None
    
    text = clean_text(text)

    current_position_site = extract_current_position(text)

    birth_year = extract_birth_year(text)

    phd_year = extract_phd_year(text)

    past_positions = extract_past_positions(text)

    # récupérer la dernière position académique
    latest_position = get_latest_position(past_positions)

    # extraction position depuis current_position
    position = extract_position(current_position_site)

    # si la position n'est pas trouvée dans current_position
    if position is None:
        position = latest_position

    # PRIORITÉ aux statuts CNRS / INRAE
    if latest_position == "DR CNRS / INRAE":
        position = "DR CNRS / INRAE"

    if latest_position == "CR CNRS / INRAE" and position not in ["DR CNRS / INRAE"]:
        position = "CR CNRS / INRAE"

    # maintenant on peut calculer le rang
    rank = classify_rank(position)

    if rank is None:
        rank = correct_rank(current_position_site, past_positions)

    return {
        "current_position": current_position_site,
        "position": position,
        "latest_position": latest_position,
        "birth_year": birth_year,
        "phd_year": phd_year,
        "rank": rank,
        "past_positions": past_positions
    }

In [96]:
results = []

for file in os.listdir("cv"):
    path = f"cv/{file}"
    print("Processing:", file)

    info = analyse_cv(path)

    if info is None:
        print("Skipping:", file)
        continue

    info["file"] = file
    results.append(info)

Processing: Hervé Ossard.pdf
Processing: César Hidalgo.pdf


Processing: Céline Bonnet.pdf
Processing: Ahmed Ezzeldin Mohamed.pdf
Processing: Milo Bianchi.pdf
Processing: Jérôme Renault.pdf
Processing: Alex Smolin.pdf
Processing: Emmanuel Thibault.pdf
Processing: Anouch Missirian.pdf
Processing: Mohamed Saleh.pdf
Processing: Pascal Lavergne.pdf
Processing: Jonathan Stieglitz.pdf
Processing: Bénédicte Alziary Chassat.pdf
Processing: Yassine Lefouili.pdf
Processing: Patrick Fève.pdf
Processing: Nour Meddahi.pdf
Processing: Christian Gollier.pdf
Processing: François Salanié.pdf
Processing: Thierry Magnac.pdf
Processing: Marc Ivaldi.pdf
Processing: Isis Durrmeyer.pdf
Processing: Nicolas Treich.pdf
Processing: David Alary.pdf
Processing: Abdelaati Daouia.pdf
Processing: Patrick Rey.pdf
Processing: Marion Hoffman.pdf
Processing: Christian Hellwig.pdf
Processing: Arnaud Reynaud.pdf
Processing: Jean-Paul Décamps.pdf
Processing: Andrew Rhodes.pdf
Processing: Julien Chhor.pdf
Processing: Jérôme Bolte.pdf
Processing: Augustin Tapsoba.pdf
Processing: Koen J

In [97]:
df_cv = pd.DataFrame(results)

In [98]:
df_cv["name"] = df_cv["file"].str.replace(".pdf", "", regex=False)

In [99]:
cols = ["name"] + [c for c in df_cv.columns if c != "name"]
df_cv = df_cv[cols]

In [142]:
df_cv[df_cv["position"].isna()][
    ["name","current_position","latest_position","past_positions", "rank"]
]

,name,current_position,latest_position,past_positions,rank


In [111]:
df_cv["position"].value_counts()

position
Full Professor                37
DR CNRS / INRAE               10
Assistant Professor            7
Professeur des universités     7
Pas clair / à déterminer       3
Associate Professor            2
CR CNRS / INRAE                1
Name: count, dtype: int64

In [113]:
df_tse

,name,appelation_site,role,profile_url,profession,rang
0,Angélique Acquatella,"Professeure assistante, TSE",Chercheuse TSE,https://www.tse-fr.eu/fr/people/angelique-acquatella,None,None
1,David Alary,"Professeur et Doyen Adjoint Pôle Formation, TSE",Chercheur TSE,https://www.tse-fr.eu/fr/people/david-alary,None,None
2,Ingela Alger,"Directeur de recherche, CNRS",Chercheur IAST/TSE,https://www.tse-fr.eu/fr/people/ingela-alger,DR CNRS,A
3,Bénédicte Alziary Chassat,"Professeur, TSE",Chercheur TSE,https://www.tse-fr.eu/fr/people/benedicte-alziary-chassat,None,None
4,Stefan Ambec,"Directeur de recherche, INRAe",Chercheur TSE,https://www.tse-fr.eu/fr/people/stefan-ambec,DR INRAE,A
5,Jean-Pierre Amigues,"Directeur de recherche, INRAe",Chercheur TSE,https://www.tse-fr.eu/fr/people/jean-pierre-amigues,DR INRAE,A
6,Henrik Andersson,"Professeur associé, TSE",Chercheur TSE,https://www.tse-fr.eu/fr/people/henrik-andersson,None,None
7,Tiziana Assenza,"Professeure associée d'Economie, TSE",Chercheuse TSE,https://www.tse-fr.eu/fr/people/tiziana-assenza,None,None
8,Andrea Attar,"Chargé de recherche, CNRS",Chercheur TSE,https://www.tse-fr.eu/fr/people/andrea-attar,CR CNRS,B
9,Emmanuelle Auriol,"Professeur d'Economie, TSE",Chercheur TSE,https://www.tse-fr.eu/fr/people/emmanuelle-auriol,None,None


In [112]:
df_cv[df_cv["position"]=="Assistant Professor"]

,name,current_position,position,latest_position,birth_year,phd_year,rank,past_positions,file
3,Ahmed Ezzeldin Mohamed,"assistant professor of political science, aug. 2023-current",Assistant Professor,Assistant Professor,None,NaN,B,"[{'year': 2023, 'text': 'assistant professor of political science, aug. 2023-current'}, {'year': 2018, 'text': 'lead researcher, life makers foundation, alexandria, may 2018 – sept. 2019'}, {'year': 2013, 'text': 'researcher, tusiad competitiveness forum, istanbul, sept. 2013 – sept. 2014'}]",Ahmed Ezzeldin Mohamed.pdf
6,Alex Smolin,"2019 – present, assistant professor, toulouse school of economics",Assistant Professor,Assistant Professor,birth: 09/09/1988,2016.0,B,"[{'year': 2019, 'text': '2019 – present, assistant professor, toulouse school of economics'}]",Alex Smolin.pdf
8,Anouch Missirian,"from2021: toulouse school of economics,france. assistant professorofeconomics.",Assistant Professor,Assistant Professor,None,2016.0,B,"[{'year': 2021, 'text': 'from2021: toulouse school of economics,france. assistant professorofeconomics.'}, {'year': 2020, 'text': '2020–2021: ucsantabarbara,usa.postdoctoralscholaratemlab(environmentalmarketsolutionslab).'}]",Anouch Missirian.pdf
30,Julien Chhor,"assistant professor. toulouse, france",Assistant Professor,None,None,NaN,B,[],Julien Chhor.pdf
32,Augustin Tapsoba,"2019- assistant professor of economics, toulouse school of economics (tse)",Assistant Professor,None,born in 2021,NaN,B,"[{'year': 2019, 'text': '2019- assistant professor of economics, toulouse school of economics (tse)'}, {'year': 2024, 'text': '2024- invited researcher, the abdul latif jameel poverty action lab (j-pal), mit'}, {'year': 2024, 'text': '2024: anr team research grant (e500 000, researcher in the project of pi nour meddahi)'}]",Augustin Tapsoba.pdf
48,Eugenia Gonzalez-Aguado,"toulouse school of economics, assistant professor 2020–current",Assistant Professor,Assistant Professor,None,NaN,B,"[{'year': 2020, 'text': 'toulouse school of economics, assistant professor 2020–current'}]",Eugenia Gonzalez-Aguado.pdf
55,Catherine Molho,"assistant professor,toulouse school of economics",Assistant Professor,Assistant Professor,None,2019.0,B,"[{'year': 2024, 'text': '2024‑ assistant professor,departmentofsocialandbehavioralsciences,toulouse school of economics'}, {'year': 2022, 'text': '2022‑2024 assistant professor,departmentofexperimentalandappliedpsychology,vuamsterdam'}, {'year': 2017, 'text': '•committeememberforbestdissertationaward,dutchassociationofsocialpsychologyresearchers(aspo)(2017‑2018)'}]",Catherine Molho.pdf


In [117]:
df_cv.columns

Index(['name', 'current_position', 'position', 'latest_position', 'birth_year',
       'phd_year', 'rank', 'past_positions', 'file'],
      dtype='object')

In [149]:
df_merge = df_tse.merge(
    df_cv[["name","position","rank"]],
    on="name",
    how="left"
)

In [150]:
df_merge["profession"] = df_merge["profession"].fillna(df_merge["position"])
df_merge["rang"] = df_merge["rang"].fillna(df_merge["rank"])

In [151]:
df_merge = df_merge.drop(columns=["position","rank"])

In [152]:
df_merge

,name,appelation_site,role,profile_url,profession,rang
0,Angélique Acquatella,"Professeure assistante, TSE",Chercheuse TSE,https://www.tse-fr.eu/fr/people/angelique-acquatella,Assistant Professor,B
1,David Alary,"Professeur et Doyen Adjoint Pôle Formation, TSE",Chercheur TSE,https://www.tse-fr.eu/fr/people/david-alary,Associate Professor,A
2,Ingela Alger,"Directeur de recherche, CNRS",Chercheur IAST/TSE,https://www.tse-fr.eu/fr/people/ingela-alger,DR CNRS,A
3,Bénédicte Alziary Chassat,"Professeur, TSE",Chercheur TSE,https://www.tse-fr.eu/fr/people/benedicte-alziary-chassat,Professeur des universités,A
4,Stefan Ambec,"Directeur de recherche, INRAe",Chercheur TSE,https://www.tse-fr.eu/fr/people/stefan-ambec,DR INRAE,A
5,Jean-Pierre Amigues,"Directeur de recherche, INRAe",Chercheur TSE,https://www.tse-fr.eu/fr/people/jean-pierre-amigues,DR INRAE,A
6,Henrik Andersson,"Professeur associé, TSE",Chercheur TSE,https://www.tse-fr.eu/fr/people/henrik-andersson,Full Professor,A
7,Tiziana Assenza,"Professeure associée d'Economie, TSE",Chercheuse TSE,https://www.tse-fr.eu/fr/people/tiziana-assenza,Associate Professor,A
8,Andrea Attar,"Chargé de recherche, CNRS",Chercheur TSE,https://www.tse-fr.eu/fr/people/andrea-attar,CR CNRS,B
9,Emmanuelle Auriol,"Professeur d'Economie, TSE",Chercheur TSE,https://www.tse-fr.eu/fr/people/emmanuelle-auriol,Full Professor,A


In [153]:
import gender_guesser.detector as gender
d = gender.Detector()

In [154]:
df_merge["first_name"] = df_merge["name"].str.split().str[0]

In [155]:
df_merge["gender_guess"] = df_merge["first_name"].apply(d.get_gender)

In [156]:
def clean_gender(g):
    if g in ["male", "mostly_male"]:
        return "Male"
    if g in ["female", "mostly_female"]:
        return "Female"
    return "Unknown"

df_merge["gender"] = df_merge["gender_guess"].apply(clean_gender)

In [157]:
df_merge["gender"].value_counts()

gender
Male       78
Female     35
Unknown     9
Name: count, dtype: int64

In [165]:
df_merge[df_merge["gender"]=="Unknown"]

,name,appelation_site,role,profile_url,profession,rang,first_name,gender_guess,gender
103,Jean-Charles Rochet,"Professeur d'Economie, TSE",Chercheur TSE,https://www.tse-fr.eu/fr/people/jean-charles-rochet,Full Professor,A,Jean-Charles,unknown,Unknown


In [166]:
gender_manual = {
    "Jean-François": "Male",
    "Jean-Charles": "Male",
    "Abdelaati": "Male",
    "Doh-Shin": "Male",
    "Chuqing": "Male",
    "Josepa": "Male",
    "Anouch": "Female",
    "Manh-Hung": "Male",
    "Soenje": "Female"
}

In [167]:
df_merge.loc[df_merge["gender"] == "Unknown", "gender"] = (
    df_merge.loc[df_merge["gender"] == "Unknown", "first_name"]
    .map(gender_manual)
    .fillna("Unknown")
)

In [168]:
df_merge["gender"].value_counts()

gender
Male      85
Female    37
Name: count, dtype: int64

In [169]:
pd.crosstab(df_merge["gender"], df_merge["rang"])

rang,A,B,Pas clair / à déterminer
gender,,,
Female,11,22,1
Male,57,23,1


In [170]:
pd.crosstab(df_merge["profession"], df_merge["gender"])

gender,Female,Male
profession,,
Assistant Professor,8,9
Associate Professor,1,1
CR CNRS,1,2
CR INRAE,3,2
DR CNRS,1,8
DR CNRS / INRAE,1,7
DR INRAE,3,3
Full Professor,4,33
Maître de conférences,10,10


In [171]:
pd.crosstab(
    df_merge["profession"],
    df_merge["gender"],
    normalize="index"
)

gender,Female,Male
profession,,
Assistant Professor,0.470588,0.529412
Associate Professor,0.500000,0.500000
CR CNRS,0.333333,0.666667
CR INRAE,0.600000,0.400000
DR CNRS,0.111111,0.888889
DR CNRS / INRAE,0.125000,0.875000
DR INRAE,0.500000,0.500000
Full Professor,0.108108,0.891892
Maître de conférences,0.500000,0.500000


# crest

In [103]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://crest.science/faculty/"

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

people = []

cards = soup.select("div.people")

for card in cards:

    # nom
    fullname = card.select_one("h3.fullname")
    name = fullname.get_text(" ", strip=True) if fullname else None

    # research interests
    interests_tag = card.select_one("div.interests span p")
    interests = interests_tag.get_text(strip=True) if interests_tag else None

    # lien profil
    link_tag = card.select_one("h3.fullname a")
    link = "https://crest.science" + link_tag["href"] if link_tag else None

    people.append({
        "name": name,
        "interests": interests,
        "profile_link": link
    })

df_crest = pd.DataFrame(people)

In [104]:
df_crest

,name,interests,profile_link
0,Marie-Laure ALLAIN,"Antitrust, Vertical Relations, Mergers, Industrial Organization",https://crest.science/user/Marie-Laure-ALLAIN
1,Stéphane AURAY,"Macroeconomics, open macro, trade, labor markets dynamics, operation research and economics of wars.",https://crest.science/user/Stéphane-AURAY
2,Geoffrey BARROWS,"Environmental Economics, International Trade",https://crest.science/user/Geoffrey-BARROWS
3,Suzanne BELLUE,"Macroeconomics, Family, Labor, Development Economics",https://crest.science/user/Suzanne-BELLUE
4,Christian BELZIL,"Microeconometrics, Education, Behavioral Economics",https://crest.science/user/Christian-BELZIL
5,David BOUNIE,"Digital Finance, Digital Economics, Industrial Organization",https://crest.science/user/David-BOUNIE
6,Marc BOURREAU,"Industrial organization, competition policy and regulation, digital economics, and telecommunications.",https://crest.science/user/Marc-BOURREAU
7,Pierre BOYER,Public Economics Political Economics Theory of Organizations Banking Regulation European integration,https://crest.science/user/Pierre-BOYER
8,Victor-Emmanuel BRUNEL,"Non-parametric statistics, Learning Theory, Geometric estimation",https://crest.science/user/Victor-Emmanuel-BRUNEL
9,Cristina BUTUCEA,"Non-parametric statistics, Learning theory, Differential Privacy, Quantum statistics",https://crest.science/user/Cristina-BUTUCEA


In [105]:
def get_profile_text(url):

    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    block = soup.select_one("div.um-name")

    if block:
        return block.get_text(" ", strip=True)

    return None

In [106]:
df_crest["profile_text"] = df_crest["profile_link"].apply(get_profile_text)

KeyboardInterrupt: 

In [ ]:
df_crest["profile_text"] = df_crest["profile_text"].str.replace(
    r"^.*?(?=CREST)", "", regex=True
)

In [ ]:
df_crest["profile_text"].value_counts()

profile_text
CREST Permanent Member Economics Personal Website                           16
CREST Permanent Member Full Professor Economics Personal Website            14
CREST Permanent Member CNRS Research Fellow Economics Personal Website      13
CREST Permanent Member Statistics Personal Website                          11
CREST Permanent Member Full Professor Statistics Personal Website           10
CREST Permanent Member Full Professor Finance Personal Website               6
CREST Permanent Member Associate Professor Statistics Personal Website       5
CREST Permanent Member Assistant Professor Economics Personal Website        5
CREST Permanent Member Assistant Professor Statistics Personal Website       4
CREST Permanent Member Associate Professor Economics Personal Website        4
CREST Permanent Member Finance Personal Website                              3
CREST Permanent Member Sociology Personal Website                            3
CREST Permanent Member CNRS Research Fe

In [ ]:
df_crest["position"] = None

In [ ]:
df_crest.loc[
    df_crest["profile_text"].str.contains("full professor", case=False, na=False),
    "position"
] = "Full Professor"

In [ ]:
df_crest.loc[
    df_crest["profile_text"].str.contains("CNRS Research Director", case=False, na=False),
    "position"
] = "DR CNRS / INRAE"

In [ ]:
df_crest.loc[
    df_crest["profile_text"].str.contains("associate professor|Associate", case=False, na=False),
    "position"
] = "Associate Professor"

In [ ]:
df_crest.loc[
    df_crest["profile_text"].str.contains("assistant professor", case=False, na=False),
    "position"
] = "Assistant Professor"

In [ ]:
df_crest["rank"] = None

In [ ]:
df_crest.loc[
    df_crest["position"].isin(["Full Professor","Associate Professor"]),
    "rank"
] = "A"

In [ ]:
df_crest.loc[
    df_crest["position"] == "Assistant Professor",
    "rank"
] = "B"

In [ ]:
len(df_crest[df_crest["position"].notna()])

51

## on s'occupe de ceux qui n'ont pas encore de position

In [ ]:
def get_bio_text(url):

    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    bio = soup.select_one('div[data-key="description"] .um-field-value')

    if bio:
        bio_text = bio.get_text(separator=" ", strip=True)
    else:
        bio_text = None

    return bio_text

In [ ]:
df_crest["bio_text"] = None

mask = df_crest["position"].isna()

df_crest.loc[mask, "bio_text"] = df_crest.loc[mask, "profile_link"].apply(get_bio_text)

In [ ]:
# DR
df_crest.loc[
    df_crest["bio_text"].str.contains(
        "research director|senior research fellow|senior researcher|Senior Resarch fellow|Senior Research",
        case=False, na=False
    ),
    "position"
] = "DR CNRS / INRAE"


# CR
df_crest.loc[
    df_crest["bio_text"].str.contains(
        r"Junior Researcher|chargé de recherche|\bcr\b",
        case=False, na=False
    ),
    "position"
] = "CR CNRS / INRAE"


# Professors seulement si pas CNRS
df_crest.loc[
    df_crest["position"].isna() &
    ~df_crest["profile_text"].str.contains("CNRS|INRAE", case=False, na=False) &
    df_crest["bio_text"].str.contains("full professor", case=False, na=False),
    "position"
] = "Full Professor"


df_crest.loc[
    df_crest["position"].isna() &
    ~df_crest["profile_text"].str.contains("CNRS|INRAE", case=False, na=False) &
    df_crest["bio_text"].str.contains(r"\bassociate\s+professor\b", case=False, na=False),
    "position"
] = "Associate Professor"


df_crest.loc[
    df_crest["position"].isna() &
    ~df_crest["profile_text"].str.contains("CNRS|INRAE", case=False, na=False) &
    df_crest["bio_text"].str.contains("assistant professor", case=False, na=False),
    "position"
] = "Assistant Professor"

In [ ]:
df_crest.loc[
    df_crest["position"].isin(["Full Professor","Associate Professor"]),
    "rank"
] = "A"

df_crest.loc[
    df_crest["position"].isin(["Assistant Professor","CR CNRS / INRAE"]),
    "rank"
] = "B"

df_crest.loc[
    df_crest["position"] == "DR CNRS / INRAE",
    "rank"
] = "A"

In [ ]:
df_crest[df_crest["name"].str.contains("TAUGOURDEAU")]

,name,interests,profile_link,profile_text,position,rank,bio_text
93,Emmanuelle TAUGOURDEAU,"Public Economics, Environmental Economics, App...",https://crest.science/user/Emmanuelle-TAUGOURDEAU,CREST Permanent Member CNRS Research Fellow Ec...,None,None,I am CNRS Research Fellow at CREST since 2017 ...


In [ ]:
df_crest["position"].isna().sum()

np.int64(42)

In [ ]:
df_crest["position"].notna().sum()

np.int64(65)

In [ ]:
df_crest[df_crest["position"].isna()].head()

,name,interests,profile_link,profile_text,position,rank,bio_text
2,Geoffrey BARROWS,"Environmental Economics, International Trade",https://crest.science/user/Geoffrey-BARROWS,CREST Permanent Member CNRS Research Fellow Ec...,None,None,I am a CNRS researcher affiliated at the Ecole...
3,Suzanne BELLUE,"Macroeconomics, Family, Labor, Development Eco...",https://crest.science/user/Suzanne-BELLUE,CREST Permanent Member Economics Personal Website,None,None,None
6,Marc BOURREAU,"Industrial organization, competition policy an...",https://crest.science/user/Marc-BOURREAU,CREST Permanent Member Economics Personal Website,None,None,Marc Bourreau is Professor of Economics at Tel...
10,Olivier CATONI,"Statistical Learning Theory, PAC-Bayes bounds,...",https://crest.science/user/Olivier-CATONI,CREST Permanent Member CNRS Research Fellow St...,None,None,None
12,Béatrice CHERRIER,History of economics,https://crest.science/user/Béatrice-CHERRIER,CREST Permanent Member CNRS Research Fellow Ec...,None,None,I am a historian of economics. I am a CNRS res...


# # récupérer le site web de ceux qu'on a tjs pas 

In [ ]:
def get_personal_website(url):

    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    link = soup.find("a", string="Personal Website")

    if link:
        return link["href"]
    
    return None

In [ ]:
def get_site_text(url):

    if pd.isna(url):
        return None

    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        r = requests.get(url, headers=headers, timeout=10)

        soup = BeautifulSoup(r.text, "html.parser")

        text = soup.get_text(separator=" ", strip=True)

        return text.lower()

    except:
        return None

In [ ]:
mask = df_crest["position"].isna()

df_crest.loc[mask, "personal_website"] = df_crest.loc[mask, "profile_link"].apply(get_personal_website)

In [ ]:
df_crest.loc[mask, "site_text"] = df_crest.loc[mask, "personal_website"].apply(get_site_text)

In [ ]:
# DR
df_crest.loc[
    df_crest["position"].isna() &
    df_crest["site_text"].str.contains(
        r"\bresearch director\b|\bdirecteur de recherche\b",
        case=False, na=False
    ),
    "position"
] = "DR CNRS / INRAE"


# CR
df_crest.loc[
    df_crest["position"].isna() &
    df_crest["site_text"].str.contains(
        r"junior researcher|chargé de recherche|\bcr\b",
        case=False, na=False
    ),
    "position"
] = "CR CNRS / INRAE"


# Maître de conférences (PRIORITÉ)
df_crest.loc[
    df_crest["position"].isna() &
    df_crest["site_text"].str.contains(
        r"ma[iî]tre de conf[eé]rences",
        case=False, na=False
    ),
    "position"
] = "Maître de conférences"


# Full Professor
df_crest.loc[
    df_crest["position"].isna() &
    ~df_crest["site_text"].str.contains("cnrs|inrae", case=False, na=False) &
    df_crest["site_text"].str.contains("full professor", case=False, na=False),
    "position"
] = "Full Professor"


# Associate Professor
df_crest.loc[
    df_crest["position"].isna() &
    ~df_crest["site_text"].str.contains("cnrs|inrae", case=False, na=False) &
    df_crest["site_text"].str.contains(r"\bassociate\s+professor\b", case=False, na=False),
    "position"
] = "Associate Professor"


# Assistant Professor
df_crest.loc[
    df_crest["position"].isna() &
    df_crest["site_text"].str.contains(
        r"\bassistant\s+professor\b",
        case=False, na=False
    ),
    "position"
] = "Assistant Professor"

In [ ]:
df_crest.loc[
    df_crest["position"].isin(["Full Professor","Associate Professor","DR CNRS / INRAE"]),
    "rank"
] = "A"

df_crest.loc[
    df_crest["position"].isin(["Assistant Professor","CR CNRS / INRAE"]),
    "rank"
] = "B"

In [ ]:
df_crest["profile_text"].value_counts()

profile_text
CREST Permanent Member Economics Personal Website                           16
CREST Permanent Member Full Professor Economics Personal Website            14
CREST Permanent Member CNRS Research Fellow Economics Personal Website      13
CREST Permanent Member Statistics Personal Website                          11
CREST Permanent Member Full Professor Statistics Personal Website           10
CREST Permanent Member Full Professor Finance Personal Website               6
CREST Permanent Member Associate Professor Statistics Personal Website       5
CREST Permanent Member Assistant Professor Economics Personal Website        5
CREST Permanent Member Assistant Professor Statistics Personal Website       4
CREST Permanent Member Associate Professor Economics Personal Website        4
CREST Permanent Member Finance Personal Website                              3
CREST Permanent Member Sociology Personal Website                            3
CREST Permanent Member CNRS Research Fe

In [ ]:
df_crest = df_crest[~df_crest["profile_text"].str.contains("sociology", case=False, na=False)]

In [ ]:
df_crest = df_crest[~df_crest["profile_text"].str.contains("finance", case=False, na=False)]

In [ ]:
df_crest["position"].isna().sum()

np.int64(16)

### récuparation de ceux qui nous manquent par recherche internet

In [ ]:
df_crest[df_crest["position"].isna()].head()

,name,interests,profile_link,profile_text,position,rank,bio_text,personal_website,site_text
2,Geoffrey BARROWS,"Environmental Economics, International Trade",https://crest.science/user/Geoffrey-BARROWS,CREST Permanent Member CNRS Research Fellow Ec...,None,None,I am a CNRS researcher affiliated at the Ecole...,https://sites.google.com/site/geoffreybarrows/,geoffrey barrows search this site embedded fil...
10,Olivier CATONI,"Statistical Learning Theory, PAC-Bayes bounds,...",https://crest.science/user/Olivier-CATONI,CREST Permanent Member CNRS Research Fellow St...,None,None,None,http://www.crest.fr/pagesperso.php?user=3435,page not found - crest latest news march 8: ce...
12,Béatrice CHERRIER,History of economics,https://crest.science/user/Béatrice-CHERRIER,CREST Permanent Member CNRS Research Fellow Ec...,None,None,I am a historian of economics. I am a CNRS res...,https://beatricecherrier.wordpress.com/about/,about beatrice cherrier – the undercover histo...
26,Alexandra DE GENDRE,None,https://crest.science/user/Alexandra-DE GENDRE,CREST Permanent Member CNRS Research Fellow Ec...,None,None,I am an economist interested in applied microe...,https://adegendre.github.io/,alexandra de gendre alexandra de gendre home r...
38,Javier GONZALEZ DELGADO,"Selective inference, hypothesis testing, clust...",https://crest.science/user/Javier-GONZALEZ DEL...,None,None,None,None,None,None


In [ ]:
pip install ddgs

/opt/python/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=37805) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from ddgs import DDGS

def search_first_result(name):

    query = f"{name} economics professor"

    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=10))

    for r in results:
        url = r["href"].lower()

        # exclure complètement certaines sources
        if any(bad in url for bad in [
            "crest.science",
            "linkedin",
            "hec.edu",
            "researchgate",
            "scholar.google",
            "wikipedia",
            ".pdf"
        ]):
            continue

        # privilégier les sites personnels académiques
        if any(domain in url for domain in [
            "github.io",
            "sites.google",
            "personal",
            "faculty",
            "people",
            "staff",
            "telecom-paris",
            "ensae",
            "polytechnique"
        ]):
            return url

    # fallback : premier résultat acceptable
    for r in results:
        url = r["href"].lower()

        if not any(bad in url for bad in [
            "crest.science",
            "linkedin",
            "hec.edu",
            "researchgate",
            "scholar.google",
            "wikipedia",
            ".pdf"
        ]):
            return url

    return None

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_page_text(url):

    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        text = soup.get_text(" ", strip=True).lower()

        return text

    except:
        return None

In [ ]:
mask = df_crest["position"].isna()

In [ ]:
df_crest.loc[mask, "google_site"] = df_crest.loc[mask, "name"].apply(search_first_result)

In [ ]:
df_crest.loc[mask, "google_text"] = df_crest.loc[mask, "google_site"].apply(get_page_text)

In [ ]:
# DR
df_crest.loc[
    df_crest["position"].isna() &
    df_crest["google_text"].str.contains(
        r"\bresearch director\b|\bdirecteur de recherche\b",
        case=False, na=False
    ),
    "position"
] = "DR CNRS / INRAE"


# CR
df_crest.loc[
    df_crest["position"].isna() &
    df_crest["google_text"].str.contains(
        r"junior researcher|chargé de recherche|\bcr\b",
        case=False, na=False
    ),
    "position"
] = "CR CNRS / INRAE"


# Maître de conférences
df_crest.loc[
    df_crest["position"].isna() &
    df_crest["google_text"].str.contains(
        r"ma[iî]tre de conf[eé]rences",
        case=False, na=False
    ),
    "position"
] = "Maître de conférences"


# Professor (poste actuel)
df_crest.loc[
    df_crest["position"].isna() &
    ~df_crest["site_text"].str.contains(
        r"assistant\s+professor|associate\s+professor",
        case=False, na=False
    ) &
    df_crest["site_text"].str.contains(
        r"\bprofessor\b",
        case=False, na=False
    ),
    "position"
] = "Full Professor"


# Associate Professor
df_crest.loc[
    df_crest["position"].isna() &
    ~df_crest["google_text"].str.contains("cnrs|inrae", case=False, na=False) &
    df_crest["google_text"].str.contains(
        r"\bassociate\s+professor\b",
        case=False, na=False
    ),
    "position"
] = "Associate Professor"


# Assistant Professor
df_crest.loc[
    df_crest["position"].isna() &
    df_crest["google_text"].str.contains(
        r"\bassistant\s+professor\b|\bassistant\s+professorship\b",
        case=False, na=False
    ),
    "position"
] = "Assistant Professor"

In [ ]:
df_crest.loc[mask, ["name", "position", "google_site", "google_text"]]

name             position  \
2             Geoffrey BARROWS                 None   
10              Olivier CATONI                 None   
12           Béatrice CHERRIER                 None   
26         Alexandra DE GENDRE                 None   
38     Javier GONZALEZ DELGADO                 None   
43               Cédric HERZET  Associate Professor   
45           Guillaume HOLLARD       Full Professor   
50               Fanny LANDAUD                 None   
59          Guillaume MAILLARD  Assistant Professor   
67              Antonio OCELLO  Assistant Professor   
77                 Julien PRAT                 None   
80              Giovanni RICCO       Full Professor   
83             Enrico RUBOLINO  Assistant Professor   
98   Clémentine VAN EFFENTERRE  Assistant Professor   
100             Michael VISSER                 None   
101         Patrick WAELBROECK                 None   

                                                                                              google_site  \
2                                                          https://sites.google.com/site/geoffreybarrows/   
10                                          https://researchportal.ip-paris.fr/en/persons/olivier-catoni/   
12                                             https://stonecenter.uchicago.edu/people/beatrice-cherrier/   
26                                                                           https://adegendre.github.io/   
38                                                                             https://jjgecon.github.io/   
43                                                                             https://cherzet.github.io/   
45                                                          https://sites.google.com/site/webpagehollard/   
50                                                        https://sites.google.com/view/fannylandaud/home   
59                                                      https://www.normalesup.org/~maillard/main_en.html   
67                                                                      https://antonio-ocello.github.io/   
77                                                      https://sites.google.com/site/julienpratecon/home   
80                                                           https://cepr.org/about/people/giovanni-ricco   
83                                                           https://sites.google.com/view/enricorubolino   
98                                             https://sites.google.com/site/vaneffenterreclementine/home   
100  https://www.calstate.edu/csu-system/faculty-staff/outstanding-faculty/pages/michael-visser-ph-d.aspx   
101                                                  https://www.telecom-paris.fr/patrick-waelbroeck?l=en   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [ ]:
df_crest.loc[
    df_crest["position"].isin(["Full Professor","Associate Professor","DR CNRS / INRAE"]),
    "rank"
] = "A"

df_crest.loc[
    df_crest["position"].isin(["Assistant Professor","CR CNRS / INRAE"]),
    "rank"
] = "B"

In [ ]:
len(df_crest[df_crest["position"].isna()])

9

In [ ]:
df_crest["position"].value_counts()

position
Full Professor           28
Assistant Professor      26
Associate Professor      15
DR CNRS / INRAE           8
CR CNRS / INRAE           2
Maître de conférences     1
Name: count, dtype: int64

In [ ]:
df_crest[df_crest["position"] == "Assistant Professor"]

name  \
3                Suzanne BELLUE   
17                 Julien COMBE   
28                Vincent DIVOL   
30              Salima EL-KOLEI   
32                 Michele FABI   
33                   Yameng FAN   
34                Johann FAOUZI   
41        Florian GROSSET-TOUBA   
42         Sébastien HERBRETEAU   
48                   Anna KORBA   
51                 Elia LAPENTA   
55             Marion LEROUTIER   
59           Guillaume MAILLARD   
64              Jaouad MOURTADA   
65   Mohammadreza MOUSAVI-KALAN   
67               Antonio OCELLO   
69                   Louis PAPE   
73              Bertille PICARD   
74             Emmanuel PILLIAT   
83              Enrico RUBOLINO   
87              Sara SIGNORELLI   
89              Ludovic STEPHAN   
90               Austin STROMME   
91                  Yuki TAMURA   
98    Clémentine VAN EFFENTERRE   
103                 Ikko YAMANE   

                                                                                                               interests  \
3                                                                   Macroeconomics, Family, Labor, Development Economics   
17                                                                               Market Design, Matching, Applied Theory   
28                                                                                                                  None   
30     Bayesian Statistics, Monte Carlo Particle Methods, time series analysis, hidden Markov models, state-space models   
32                                                                                  Microeconomics, Financial Economics.   
33                                                                                                                  None   
34   Machine learning, open source software, Python programming, time series and signal processing, medical applications   
41                                                       Development economics, environmental economics, labor economics   
42                                                                                                                  None   
48                                                                     Machine Learning, Optimal Transport, Optimisation   
51                                                                                              Theoretical econometrics   
55                                                                       Environmental Economics, Applied Microeconomics   
59                                                                                                                  None   
64                                                              Statistical learning theory, High-dimensional statistics   
65        Statistical machine learning, Transfer learning, Domain adaptation, Optimization Theory, Distrusted Computing.   
67                                                                                                                  None   
69                                                 Digital economics, Empirical Industrial Organization, Labor Economics   
73                                                                                                                  None   
74                                                                                                                  None   
83                                                                                                                  None   
87                                                                      Labor Economics, Migration Economics, Innovation   
89                                                                                                                  None   
90                                    Statistical optimal transport, Riemannian optimization, theory of machine learning   
91                                                                         Market Design, Social Choice, 

# pse

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.parisschoolofeconomics.eu/en/people/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
}

r = requests.get(url, headers=headers)

soup = BeautifulSoup(r.text, "html.parser")

print(r.status_code)

200


In [ ]:
print("Piketty" in r.text)

False


# lire un html pour les 5 sites

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.parisschoolofeconomics.eu/personnes/?keyword=&_persons-category-thematics=23143&_persons-category-thematics=23143&_persons-category-thematics=23228&_persons-category-thematics=23222&_persons-category-thematics=23148&_persons-category-thematics=23217&_persons-category-thematics=23203&_persons-category-thematics=23228%2C23222%2C23148%2C23217%2C23203"

headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
response.encoding = "utf-8"

html = response.text

soup = BeautifulSoup(html, "html.parser")

pretty_html = soup.prettify()

with open("page.html", "w", encoding="utf-8") as f:
    f.write(pretty_html)